# Lezione 16 - Distribuire agenti scalabili con Microsoft Foundry

In questo notebook costruisci un **agente di supporto clienti pronto per la produzione** per l'azienda fittizia **Contoso**. A differenza delle lezioni precedenti, il punto qui non è il ciclo di ragionamento dell'agente — è tutto ciò che lo circonda che rende un agente sicuro da eseguire su larga scala:

1. **Chiamata agli strumenti** — ricerche d'ordine e creazione ticket.
2. **RAG** — risposte policy da una base di conoscenza.
3. **Memoria** — ricordare il cliente attraverso i turni.
4. **Instradamento del modello** — inviare richieste semplici a un modello piccolo, quelle complesse a un modello grande.
5. **Caching delle risposte** — servire domande ripetute senza chiamare un modello.
6. **Approvazione umana** — rimborsi sopra una soglia in pausa per l'approvazione.
7. **Blocco di valutazione** — un set di test offline che blocca una cattiva release.
8. **Osservabilità** — tracing OpenTelemetry attorno a ogni richiesta.

Ogni sezione è autonoma ed eseguibile. Leggi ogni riga — le primitive di produzione sono mantenute volutamente piccole.


## Configurazione

Prima di eseguire questo notebook, assicurati di avere:

1. **Un progetto Microsoft Foundry** con un modello di chat distribuito (ad esempio `gpt-5-mini`).
2. **Accesso effettuato con Azure CLI** — esegui `az login` nel terminale.
3. **Impostate le variabili d'ambiente necessarie:**
   - `AZURE_AI_PROJECT_ENDPOINT` — il punto di accesso del tuo progetto Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — il nome del modello distribuito.

La sezione RAG utilizza **Azure AI Search** quando sono impostati `AZURE_SEARCH_SERVICE_ENDPOINT` e `AZURE_SEARCH_API_KEY`, e ricorre a una ricerca in memoria in modo che il notebook funzioni anche senza una risorsa Search.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Strumenti

Gli strumenti di produzione svolgono un lavoro reale su sistemi reali. Qui simuliamo un database di ordini e un sistema di ticketing con semplici funzioni Python. Il decoratore `@tool` li espone all'agente.

Nota che `issue_refund` utilizza `approval_mode="always_require"` per rimborsi sopra una certa soglia — questa è la primitiva human-in-the-loop che implementiamo in seguito.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Base di Conoscenza delle Politiche

Le domande sulle politiche ("qual è la tua finestra di reso?") dovrebbero essere risposte da una fonte autorevole, non dalla memoria del modello. Avvolgiamo un piccolo database di conoscenza come strumento di ricerca.

In produzione questo è **Azure AI Search**; qui forniamo una ricerca per parole chiave in memoria così che il notebook funzioni ovunque e si passi automaticamente a Azure AI Search quando sono presenti le variabili d'ambiente.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Memoria

Un agente di supporto che dimentica con chi sta parlando è un cattivo agente di supporto. Conserviamo un piccolo archivio di profili per cliente e inseriamo un breve riassunto nelle istruzioni dell'agente. In produzione questo è un servizio di memoria (vedi Lezione 13); qui un dizionario rende il modello visibile.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 e 5. Routing del Modello e Caching delle Risposte

Due leve di costo collegate in un unico gestore di richieste:

- **Routing**: un classificatore euristico economico decide se una richiesta necessita del modello piccolo o grande.
- **Caching**: domande ripetute normalizzate vengono servite direttamente da una cache senza chiamata al modello.

Il classificatore qui è intenzionalmente semplice. In produzione lo convalideresti con il traffico e potresti sostituirlo con il Model Router di Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. L'Agente, l'Approvazione Umana e l'Osservabilità

Ora assemblamo l'agente dagli strumenti sopra e avvolgiamo ogni richiesta in uno span OpenTelemetry. La funzione `handle_support_request` è il gestore delle richieste in produzione: cache → route → trace → run → cache.

L'approvazione umana è gestita dal framework: poiché `issue_refund` ha `approval_mode="always_require"`, l'esecuzione si mette in pausa e mostra una richiesta di approvazione che risolviamo esplicitamente.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Porta di Valutazione

Questa è la porta di rilascio della lezione: un set di test offline valuta l'agente e la distribuzione procede solo se la percentuale di superamento supera una soglia. Il valutatore qui è un semplice controllo di sovrapposizione di parole chiave per mantenere il notebook autonomo; in produzione si userebbe un LLM-giudice o un valutatore di framework (vedi Lezione 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Mettere Tutto Insieme: Una Rilascio Simulato

La cella qui sotto mostra l'intero ciclo descritto nella lezione: esegui il gate di valutazione, e "deploy" solo se viene superato. Questo è il modello che useresti in CI prima di promuovere una versione dell'agente al Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Sommario

Hai assemblato un agente di supporto clienti pronto per la produzione con ogni aspetto operativo integrato:

- **Strumenti, RAG e memoria** forniscono capacità e contesto all'agente.
- **Instradamento del modello e caching** mantengono sotto controllo latenza e costi.
- **Approvazione umana** protegge azioni ad alto rischio come grandi rimborsi.
- **Il cancello di valutazione** blocca le release difettose prima della distribuzione.
- **Tracciamento** rende ogni richiesta osservabile.

### Sfida

Estendi questo agente per:

1. **Supportare modelli multipli** — aggiungi un terzo livello di "ragionamento" e indirizza a questo le escalation/riclami.
2. **Aggiungere cancelli di valutazione** — espandi `TEST_CASES` includendo scenari di approvazione rimborso e conferma che il cancello intercetti regressioni.
3. **Aggiungere instradamento consapevole del costo** — traccia un costo stimato per richiesta (piccola vs grande vs cache) e stampa un rapporto dei costi dopo un batch di query miste.

Nella lezione successiva intraprenderai il percorso opposto e gestirai un agente interamente sulla tua macchina con Microsoft Foundry Local e Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
